# 69-claim neural CFR+ branch screen: full early expansion

This notebook branches from an existing `r6_s4_h4_hp2tfhq_ss` neural CFR+ checkpoint and tests whether fully expanding the first one or two traverser decisions improves the continuation.

It intentionally does not save full resume checkpoints for branches. It saves only small playable average-policy snapshots and JSONL diagnostics. Run fitted-return BR separately against the final snapshots.

In [ ]:
from datetime import datetime, timezone
import json
import math
from pathlib import Path
import re
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'liars_poker').is_dir():
    raise RuntimeError('Could not locate repository root.')

import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.serialization import save_policy

assert torch.cuda.is_available(), 'This branch screen is intended for a CUDA machine.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('repo:', REPO_ROOT)
print('torch:', torch.__version__)
print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
SPEC = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)

# Leave as None to use the newest matching run with latest_checkpoint.pt.
SOURCE_RUN_DIR = None

BRANCH_MINUTES = 60.0
SNAPSHOT_EVERY_MINUTES = 30.0
OUTPUT_ROOT = REPO_ROOT / 'artifacts' / 'cfr_plus_69_claim_branch_screens'

VARIANTS = [
    {
        'label': 'control 256; cap16',
        'traversals_per_player': 256,
        'regret_steps': 24,
        'strategy_steps': 6,
        'schedule': (16,),
        'traversal_batch_size': 64,
    },
    {
        'label': '256; full first then cap16',
        'traversals_per_player': 256,
        'regret_steps': 24,
        'strategy_steps': 6,
        'schedule': (69, 16),
        'traversal_batch_size': 32,
    },
    {
        'label': '256; full first two then cap16',
        'traversals_per_player': 256,
        'regret_steps': 24,
        'strategy_steps': 6,
        'schedule': (69, 69, 16),
        'traversal_batch_size': 16,
    },
    {
        'label': 'control 512; cap16',
        'traversals_per_player': 512,
        'regret_steps': 48,
        'strategy_steps': 12,
        'schedule': (16,),
        'traversal_batch_size': 64,
    },
    {
        'label': '512; full first then cap16',
        'traversals_per_player': 512,
        'regret_steps': 48,
        'strategy_steps': 12,
        'schedule': (69, 16),
        'traversal_batch_size': 32,
    },
    {
        'label': '512; full first two then cap16',
        'traversals_per_player': 512,
        'regret_steps': 48,
        'strategy_steps': 12,
        'schedule': (69, 69, 16),
        'traversal_batch_size': 16,
    },
]

pd.DataFrame(VARIANTS)

In [ ]:
def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return list(value)
    if hasattr(value, 'item'):
        return value.item()
    return str(value)


def append_jsonl(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, default=json_default) + '\n')


def read_jsonl(path):
    if not Path(path).exists():
        return []
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]


def slug(label):
    return re.sub(r'[^a-zA-Z0-9_]+', '_', label).strip('_').lower()


def find_source_run():
    short = SPEC.to_short_str()
    candidates = []
    for checkpoint in (REPO_ROOT / 'artifacts').rglob('latest_checkpoint.pt'):
        run_dir = checkpoint.parent
        if short not in run_dir.name:
            continue
        if not (run_dir / 'manifest.json').exists():
            continue
        candidates.append(run_dir)
    if not candidates:
        raise FileNotFoundError(f'No matching {short} run with latest_checkpoint.pt under artifacts/.')
    return max(candidates, key=lambda p: (p / 'latest_checkpoint.pt').stat().st_mtime)


SOURCE_RUN_DIR = Path(SOURCE_RUN_DIR) if SOURCE_RUN_DIR is not None else find_source_run()
CHECKPOINT_PATH = SOURCE_RUN_DIR / 'latest_checkpoint.pt'
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
BRANCH_ROOT = OUTPUT_ROOT / f'{SPEC.to_short_str()}___{run_id}'
BRANCH_ROOT.mkdir(parents=True, exist_ok=True)

manifest = {
    'run_type': 'cfr_plus_69_claim_branch_screen',
    'spec': SPEC.to_json(),
    'source_run_dir': str(SOURCE_RUN_DIR),
    'source_checkpoint': str(CHECKPOINT_PATH),
    'branch_minutes': BRANCH_MINUTES,
    'snapshot_every_minutes': SNAPSHOT_EVERY_MINUTES,
    'variants': VARIANTS,
}
(BRANCH_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=json_default), encoding='utf-8')

print('source:', SOURCE_RUN_DIR)
print('branch root:', BRANCH_ROOT)

In [ ]:
def configure_branch(trainer, variant):
    trainer.traversal_backend = 'gpu_native'
    trainer.traversal_batch_size = int(variant['traversal_batch_size'])
    trainer.regret_train_steps = int(variant['regret_steps'])
    trainer.strategy_train_steps = int(variant['strategy_steps'])
    trainer.traverser_action_sample_count = None
    trainer.traverser_action_sample_fraction = None
    trainer.traverser_action_full_first = False
    trainer.traverser_action_sample_schedule = tuple(int(x) for x in variant['schedule'])
    trainer.traverser_action_priority_count = 0
    trainer.traverser_action_baseline = 'none'
    trainer._gpu_traverser = None
    return trainer


def mean_float(values):
    values = list(values or [])
    return float(sum(values) / len(values)) if values else float('nan')


def flatten_record(label, variant, measured_s, iteration_wall_s, record):
    sampling = record.get('action_sampling', {}) or {}
    timing = record.get('timing', {}) or {}
    return {
        'configuration': label,
        'iteration': int(record['iteration']),
        'measured_training_s': float(measured_s),
        'iteration_wall_s': float(iteration_wall_s),
        'traversals_per_player': int(variant['traversals_per_player']),
        'regret_steps': int(variant['regret_steps']),
        'strategy_steps': int(variant['strategy_steps']),
        'schedule': list(variant['schedule']),
        'traversal_batch_size': int(variant['traversal_batch_size']),
        'regret_loss': mean_float(record.get('regret_loss')),
        'strategy_loss': mean_float(record.get('strategy_loss')),
        'new_regret_records': int(sum(record.get('new_regret_records', []) or [])),
        'new_strategy_records': int(sum(record.get('new_strategy_records', []) or [])),
        'traversal_s': float(timing.get('traversal_s', float('nan'))),
        'regret_training_s': float(timing.get('regret_training_s', float('nan'))),
        'strategy_training_s': float(timing.get('strategy_training_s', float('nan'))),
        'claim_edge_fraction': float(sampling.get('claim_edge_fraction', float('nan'))),
        'regret_weight_ess_fraction': float(sampling.get('regret_weight_ess_fraction', float('nan'))),
        'max_regret_weight': float(sampling.get('max_regret_weight', float('nan'))),
    }


def save_average_snapshot(trainer, branch_dir, measured_s):
    policy_dir = branch_dir / 'snapshots' / f'train_{measured_s:010.3f}_iter_{trainer.iteration:08d}'
    if not policy_dir.exists():
        save_policy(trainer.average_policy(), str(policy_dir))
    return policy_dir


summaries = []
budget_s = 60.0 * BRANCH_MINUTES
snapshot_period_s = 60.0 * SNAPSHOT_EVERY_MINUTES

for variant in VARIANTS:
    label = variant['label']
    branch_dir = BRANCH_ROOT / slug(label)
    branch_dir.mkdir(parents=True, exist_ok=True)
    training_path = branch_dir / 'training.jsonl'
    summary_path = branch_dir / 'summary.json'
    if summary_path.exists():
        summaries.append(json.loads(summary_path.read_text(encoding='utf-8')))
        print('skipping completed branch:', label)
        continue

    print('\n===', label, '===')
    trainer = DeepCFRPlusTrainer.load_checkpoint(CHECKPOINT_PATH, device=device)
    trainer = configure_branch(trainer, variant)
    measured_s = 0.0
    next_snapshot_s = snapshot_period_s
    last_print_s = 0.0
    snapshots = []

    while measured_s < budget_s:
        start = time.perf_counter()
        record = trainer.run_iteration(traversals_per_player=int(variant['traversals_per_player']))
        iteration_s = time.perf_counter() - start
        measured_s += iteration_s
        row = flatten_record(label, variant, measured_s, iteration_s, record)
        append_jsonl(training_path, row)

        if measured_s >= next_snapshot_s or measured_s >= budget_s:
            policy_dir = save_average_snapshot(trainer, branch_dir, measured_s)
            snapshots.append(str(policy_dir))
            print(f"snapshot {label}: train={measured_s / 60.0:.1f}m iter={trainer.iteration}")
            while next_snapshot_s <= measured_s:
                next_snapshot_s += snapshot_period_s

        if measured_s - last_print_s >= 60.0:
            last_print_s = measured_s
            print(f"{label}: train={measured_s / 60.0:.1f}m iter={trainer.iteration} iter_s={iteration_s:.3f}")

    final_policy_dir = save_average_snapshot(trainer, branch_dir, measured_s)
    rows = read_jsonl(training_path)
    df = pd.DataFrame(rows)
    summary = {
        'configuration': label,
        'branch_dir': str(branch_dir),
        'final_policy_dir': str(final_policy_dir),
        'measured_training_min': measured_s / 60.0,
        'iterations_completed': int(len(df)),
        'iterations_per_min': float(len(df) / (measured_s / 60.0)),
        'mean_traversal_s': float(df['traversal_s'].mean()),
        'mean_regret_fit_s': float(df['regret_training_s'].mean()),
        'mean_strategy_fit_s': float(df['strategy_training_s'].mean()),
        'mean_claim_edge_fraction': float(df['claim_edge_fraction'].mean()),
        'mean_regret_weight_ess_fraction': float(df['regret_weight_ess_fraction'].mean()),
        'largest_regret_weight': float(df['max_regret_weight'].max()),
    }
    summary_path.write_text(json.dumps(summary, indent=2, default=json_default), encoding='utf-8')
    summaries.append(summary)
    del trainer
    torch.cuda.empty_cache()

summary_df = pd.DataFrame(summaries).set_index('configuration')
summary_df

In [ ]:
all_rows = []
for summary in summaries:
    path = Path(summary['branch_dir']) / 'training.jsonl'
    all_rows.extend(read_jsonl(path))

df = pd.DataFrame(all_rows)
if df.empty:
    raise RuntimeError('No branch records found.')
df['measured_training_min'] = df['measured_training_s'] / 60.0

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for label, group in df.groupby('configuration'):
    axes[0, 0].plot(group['measured_training_min'], group['iteration'], label=label)
    axes[0, 1].plot(group['measured_training_min'], group['traversal_s'], label=label)
    axes[0, 2].plot(group['measured_training_min'], group['claim_edge_fraction'], label=label)
    axes[1, 0].plot(group['measured_training_min'], group['regret_weight_ess_fraction'], label=label)
    axes[1, 1].plot(group['measured_training_min'], group['max_regret_weight'], label=label)
    axes[1, 2].plot(group['measured_training_min'], group['new_regret_records'], label=label)

axes[0, 0].set_title('Iterations completed')
axes[0, 1].set_title('Traversal seconds per iteration')
axes[0, 2].set_title('Claim-edge fraction')
axes[1, 0].set_title('Regret-weight ESS fraction')
axes[1, 1].set_title('Largest inverse-path regret weight')
axes[1, 1].set_yscale('log')
axes[1, 2].set_title('New regret records per iteration')
for ax in axes.flat:
    ax.set_xlabel('Measured branch-training minutes')
    ax.grid(True, alpha=0.3)
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

summary_df.sort_values('iterations_per_min', ascending=False)

## Next evaluation step

Use the `final_policy_dir` values from the summary table as targets for fitted-return BR. Start with 20 minutes per branch. If the full-first variants look better, run 60-minute responders on the best two and the control.